# 🛢️ Nodal Analysis & Wellbore Performance Simulator
**Advanced Agentic Production Optimization Engine (Google Colab Edition)**

Welcome to the **Nodal Analysis & Wellbore Performance Simulator**! This standalone Google Colab notebook provides a complete, industrial-grade multiphase production optimization engine in a single file. You can run all cells sequentially to evaluate wellbore hydraulics, reservoir inflow, and surface back-pressure.

### 📌 Key Features:
* **PVT & Fluid Characterization:** Black-oil PVT properties using Standing (1947/1977), Papay (1968), Beggs-Robinson (1975), and Lee-Gonzalez-Eakin (1966) correlations with robust overflow protection.
* **Multiphase Wellbore Hydraulics (VLP):** Beggs & Brill (1973/1986) and Hagedorn & Brown (1965) multiphase flow correlations with Griffith-Wallis bubble-flow correction and flow regime mapping.
* **Inflow Performance Relationship (IPR):** Supports Vogel, Fetkovitch, Jones, Standing (damage/skin), and Darcy models.
* **Nodal Analysis Solver:** Precision bisection root-finding algorithm to determine the operating flow rate ($q$) and flowing bottom-hole pressure ($P_{wf}$).
* **Interactive Plotly Visualizations:** Generate dynamic IPR/VLP intersection plots, multi-tubing sizing comparisons, and depth vs. pressure wellbore profiles right inside Colab cells!

---
### ℹ️ Test Data Notice
*The default parameter values and well trajectories used in this simulation are based on standard benchmark petroleum engineering test datasets (e.g., SPE sample data). You can modify any parameter in the interactive simulation cells below to model your specific well conditions.*


In [ ]:
# =====================================================================
# STEP 1: Install & Import Required Libraries
# =====================================================================
!pip install -q numpy scipy plotly pandas

import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import interp1d
from typing import Optional, Tuple, Dict, List

print("✅ Setup complete! All scientific and plotting libraries are ready.")


## ⚙️ Module 1: PVT & Fluid Properties
This module computes black-oil fluid properties for an oil-gas-water system at reservoir and wellbore conditions.
* **Bubble-Point & Solution GOR:** Standing (1947 / 1977)
* **Oil Formation Volume Factor ($B_o$):** Standing (1947)
* **Gas Z-Factor ($z$):** Papay (1968) explicit approximation with Sutton pseudo-critical properties
* **Oil Viscosity ($\mu_o$):** Beggs & Robinson dead oil + Chew & Connally live oil
* **Gas Viscosity ($\mu_g$):** Lee, Gonzalez & Eakin (1966) with safe exponential clamping


In [ ]:
def _safe_exp(val: float) -> float:
    """Safe exponential that clamps argument to prevent math range overflow."""
    return math.exp(max(-700.0, min(700.0, val)))


class FluidProperties:
    """
    Compute PVT properties for an oil-gas-water system.

    Parameters
    ----------
    api_gravity : float
        Stock-tank oil API gravity (°API)
    gas_sg : float
        Gas specific gravity (air = 1.0)
    water_sg : float
        Water specific gravity (fresh water = 1.0)
    gor : float
        Producing GOR at standard conditions (scf/STB)
    water_cut : float
        Water cut as a fraction (0.0 – 1.0), e.g. 0.20 for 20%
    reservoir_temp : float
        Reservoir temperature (°F)
    """

    def __init__(
        self,
        api_gravity: float = 35.0,
        gas_sg: float = 0.65,
        water_sg: float = 1.07,
        gor: float = 500.0,
        water_cut: float = 0.20,
        reservoir_temp: float = 180.0,
    ):
        self.api = api_gravity
        self.gas_sg = gas_sg
        self.water_sg = water_sg
        self.gor = gor          # total GOR (scf/STB of oil)
        self.wc = water_cut     # fraction
        self.T_res = reservoir_temp  # °F

        # Derived base properties
        self.oil_sg = 141.5 / (self.api + 131.5)
        self.rho_oil_sc = self.oil_sg * 62.4    # lb/ft³ at SC
        self.rho_water_sc = self.water_sg * 62.4
        self.rho_gas_sc = self.gas_sg * 0.0764  # lb/ft³ at SC (air ~ 0.0764 lb/ft³)

        # Bubble-point pressure (at reservoir temperature)
        self.bubble_point = self._bubble_point_standing(self.T_res, self.gor)

    # ------------------------------------------------------------------ #
    #  BUBBLE-POINT PRESSURE  (Standing, 1947)                            #
    # ------------------------------------------------------------------ #
    def _bubble_point_standing(self, T_F: float, Rs: float) -> float:
        """
        Standing (1947) bubble-point pressure correlation.
        T_F  : temperature (°F)
        Rs   : solution GOR at bubble point (scf/STB)
        Returns Pb in psia.
        """
        if Rs <= 0:
            return 14.7
        x = max(-300.0, min(300.0, 0.0125 * self.api - 0.00091 * T_F))
        Pb = 18.2 * ((Rs / max(self.gas_sg, 0.01)) ** 0.83 * (10.0 ** x) - 1.4)
        return max(14.7, min(50000.0, Pb))

    # ------------------------------------------------------------------ #
    #  SOLUTION GOR  (Standing, 1977 – inverse of bubble-point)           #
    # ------------------------------------------------------------------ #
    def solution_gor(self, P: float, T_F: float) -> float:
        """
        Solution GOR at pressure P and temperature T_F (Standing).
        Returns Rs in scf/STB.  Rs <= producing GOR always.
        """
        if P >= self.bubble_point:
            return self.gor  # all gas in solution above bubble-point
        # Standing inverse: Rs = gamma_g * ((Pb/18.2 + 1.4) / 10^x)^(1/0.83)
        x = max(-300.0, min(300.0, 0.0125 * self.api - 0.00091 * T_F))
        denom = max(1e-10, 10.0 ** x)
        Rs = self.gas_sg * ((P / 18.2 + 1.4) / denom) ** (1.0 / 0.83)
        return max(0.0, min(Rs, self.gor))

    # ------------------------------------------------------------------ #
    #  OIL FORMATION VOLUME FACTOR  (Standing, 1947)                      #
    # ------------------------------------------------------------------ #
    def oil_fvf(self, P: float, T_F: float) -> float:
        """Oil FVF Bo (RB/STB) – Standing (1947)."""
        Rs = self.solution_gor(P, T_F)
        F = Rs * (self.gas_sg / max(self.oil_sg, 0.01)) ** 0.5 + 1.25 * T_F
        Bo = 0.972 + 1.47e-4 * (max(0.0, F) ** 1.175)
        return max(1.0, min(100.0, Bo))

    # ------------------------------------------------------------------ #
    #  GAS Z-FACTOR  (Papay, 1968 – fast explicit approximation)          #
    # ------------------------------------------------------------------ #
    def gas_z_factor(self, P: float, T_F: float) -> float:
        """
        Papay (1968) explicit z-factor correlation – fast, accurate for
        typical natural gas conditions (Tpr > 1.05, Ppr < 15).
        Sutton (1985) pseudo-critical properties.
        """
        # Sutton pseudo-critical properties
        Tpc = 169.2 + 349.5 * self.gas_sg - 74.0 * self.gas_sg ** 2  # °R
        Ppc = 756.8 - 131.0 * self.gas_sg - 3.6 * self.gas_sg ** 2   # psia

        T_R  = T_F + 459.67
        Tpr  = T_R / max(Tpc, 1.0)
        Ppr  = P   / max(Ppc, 1.0)

        if Ppr < 0.001:
            return 1.0

        # Papay (1968) – explicit approximation
        exp1 = max(-300.0, min(300.0, 0.9813 * Tpr))
        exp2 = max(-300.0, min(300.0, 0.8157 * Tpr))
        z = (1.0
             - 3.52 * Ppr / max(1e-10, 10.0 ** exp1)
             + 0.274 * Ppr ** 2 / max(1e-10, 10.0 ** exp2))

        # Clamp to physically valid range
        return max(0.15, min(2.5, z))

    # ------------------------------------------------------------------ #
    #  GAS FORMATION VOLUME FACTOR                                         #
    # ------------------------------------------------------------------ #
    def gas_fvf(self, P: float, T_F: float) -> float:
        """Bg in ft³/SCF at reservoir conditions."""
        z = self.gas_z_factor(P, T_F)
        T_R = T_F + 459.67
        Bg = 0.02829 * z * T_R / max(P, 14.7)  # ft³/SCF
        return max(1e-6, Bg)

    # ------------------------------------------------------------------ #
    #  OIL VISCOSITY  (Beggs & Robinson 1975 dead; Chew & Connally 1959)  #
    # ------------------------------------------------------------------ #
    def oil_viscosity(self, P: float, T_F: float) -> float:
        """Live-oil viscosity in cp."""
        # Dead-oil viscosity (Beggs & Robinson, 1975)
        exp_val = max(-300.0, min(300.0, (3.0324 - 0.02023 * self.api) / max(T_F ** 1.163, 1e-5)))
        mu_od = max(0.01, 10.0 ** exp_val - 1.0)

        Rs = self.solution_gor(P, T_F)

        # Chew & Connally (1959) live-oil viscosity
        A = 10.715 * (Rs + 100.0) ** (-0.515)
        B = 5.44 * (Rs + 150.0) ** (-0.338)
        mu_o = A * (max(0.001, min(1e5, mu_od)) ** B)
        return max(0.1, min(10000.0, mu_o))

    # ------------------------------------------------------------------ #
    #  GAS VISCOSITY  (Lee, Gonzalez & Eakin, 1966)                       #
    # ------------------------------------------------------------------ #
    def gas_viscosity(self, P: float, T_F: float) -> float:
        """Gas viscosity in cp."""
        T_R = T_F + 459.67
        Mg = 28.97 * self.gas_sg  # molecular weight g/mol
        z = self.gas_z_factor(P, T_F)
        rho_g = P * Mg / (z * 10.73 * T_R)  # lb/ft³

        K = ((9.4 + 0.02 * Mg) * (T_R ** 1.5)) / max(1e-5, 209.0 + 19.0 * Mg + T_R)
        X = 3.5 + 986.0 / max(T_R, 1e-5) + 0.01 * Mg
        Y = 2.4 - 0.2 * X

        # Clamp density ratio and exponent to prevent math range error / overflow
        rho_ratio = max(1e-5, min(100.0, rho_g / 62.4))
        exp_arg = max(-700.0, min(700.0, X * (rho_ratio ** Y)))
        mu_g = 1e-4 * K * _safe_exp(exp_arg)
        return max(0.005, min(100.0, mu_g))

    # ------------------------------------------------------------------ #
    #  WATER VISCOSITY & FVF                                               #
    # ------------------------------------------------------------------ #
    def water_viscosity(self, T_F: float) -> float:
        """Water viscosity in cp (Van Wingen, 1950 approximation)."""
        exp_arg = max(-700.0, min(700.0, 1.003 - 1.479e-2 * T_F + 1.982e-5 * (T_F ** 2)))
        mu_w = _safe_exp(exp_arg)
        return max(0.1, min(100.0, mu_w))

    def water_fvf(self, P: float, T_F: float) -> float:
        """Water FVF Bw in RB/STB (Craft & Hawkins)."""
        Bw = (
            1.0
            + 1.21e-4 * (T_F - 60.0)
            + 1.0e-6 * (T_F - 60.0) ** 2
            - 3.33e-6 * P
        )
        return max(1.0, min(10.0, Bw))

    # ------------------------------------------------------------------ #
    #  MIXTURE PROPERTIES  (for VLP correlations)                          #
    # ------------------------------------------------------------------ #
    def liquid_holdup_input(self, P: float, T_F: float) -> dict:
        """
        Compute all mixture properties needed by the VLP correlations.

        Returns a dict with:
          rho_L          : in-situ liquid density (lb/ft³)
          rho_g          : in-situ gas density (lb/ft³)
          mu_L           : liquid viscosity (cp)
          mu_g           : gas viscosity (cp)
          sigma_L        : liquid surface tension (dyne/cm)
          lambda_L       : no-slip liquid holdup (volume fraction)
          ql_res_ratio   : liquid volume at res conditions per STB surface liquid (RB/STB)
          qg_res_ratio   : gas volume at res conditions per STB surface liquid (RB/STB)
        """
        Rs = self.solution_gor(P, T_F)
        Bo = self.oil_fvf(P, T_F)
        Bw = self.water_fvf(P, T_F)
        Bg = self.gas_fvf(P, T_F)   # ft³/SCF

        oil_frac   = 1.0 - self.wc
        water_frac = self.wc
        free_gor   = max(0.0, self.gor - Rs)  # scf/STB oil (free gas)

        # Volumetric rates at reservoir conditions per STB total surface liquid
        ql_oil   = oil_frac * Bo           # RB/STB
        ql_water = water_frac * Bw         # RB/STB
        ql_res   = ql_oil + ql_water       # RB/STB (total liquid)
        # Gas volume: oil_frac × free_gor [scf/STB oil] × Bg [ft³/scf] / 5.615 [ft³/RB]
        qg_res   = oil_frac * free_gor * Bg / 5.615  # RB/STB

        total_res = ql_res + qg_res
        lambda_L  = ql_res / max(total_res, 1e-10)

        # Phase densities at reservoir conditions
        # Oil: mass of oil + dissolved gas, divided by res volume
        rho_oil_res = (self.rho_oil_sc * oil_frac
                       + self.rho_gas_sc * Rs * oil_frac / 5.615) / max(ql_oil, 1e-10)
        rho_water_res = self.rho_water_sc / Bw
        if ql_res > 1e-10:
            rho_L = (ql_oil * rho_oil_res + ql_water * rho_water_res) / ql_res
        else:
            rho_L = self.rho_oil_sc

        # Gas density at reservoir conditions (clamped to physical upper limit ~50 lb/ft³)
        rho_g = max(0.01, min(50.0, self.rho_gas_sc / max(Bg * 5.615, 1e-6)))  # lb/ft³

        # Viscosities
        mu_oil   = self.oil_viscosity(P, T_F)
        mu_water = self.water_viscosity(T_F)
        mu_gas   = self.gas_viscosity(P, T_F)
        if ql_res > 1e-10:
            mu_L = (ql_oil * mu_oil + ql_water * mu_water) / ql_res
        else:
            mu_L = mu_oil

        # Surface tension (dyne/cm ≈ mN/m)
        sigma_oil   = max(1.0, 68.0 - 0.000015 * P - 0.5 * self.api)
        sigma_water = max(1.0, 74.0 - 0.1 * (T_F - 60.0))
        sigma_L     = oil_frac * sigma_oil + water_frac * sigma_water

        return {
            "rho_L":         max(5.0, rho_L),
            "rho_g":         max(0.01, rho_g),
            "mu_L":          max(0.05, mu_L),
            "mu_g":          max(0.005, mu_gas),
            "sigma_L":       max(1.0, sigma_L),
            "lambda_L":      max(0.001, min(1.0, lambda_L)),
            "ql_res_ratio":  max(1e-6, ql_res),
            "qg_res_ratio":  max(0.0, qg_res),
        }

    def mixture_density(self, P: float, T_F: float) -> float:
        """Mixture density (oil + water + free gas) in lb/ft³ at P, T."""
        props = self.liquid_holdup_input(P, T_F)
        lam   = props["lambda_L"]
        return lam * props["rho_L"] + (1.0 - lam) * props["rho_g"]


## 🌊 Module 2: Multiphase Wellbore Hydraulics & Pressure Loss
This module calculates the three components of pressure gradient ($\frac{dP}{dz}$) at any depth:
1. **Hydrostatic (Gravity) Gradient:** Due to fluid elevation and mixture density.
2. **Friction Gradient:** Darcy-Weisbach friction using Serghides explicit Colebrook-White approximation and two-phase holdup corrections.
3. **Acceleration Gradient:** Kinetic energy change along the tubing string.

Supports **Beggs & Brill (1973/1986)** and **Hagedorn & Brown (1965)** correlations.


In [ ]:
# Gravitational acceleration in ft/s²
G_C = 32.174      # lbm·ft / (lbf·s²)
G   = 32.174      # ft/s²


def moody_friction_factor(Re: float, roughness: float, D: float) -> float:
    """
    Moody friction factor using the Serghides (1984) explicit approximation
    to the Colebrook-White equation.

    Parameters
    ----------
    Re        : Reynolds number (dimensionless)
    roughness : absolute pipe roughness (ft)
    D         : pipe inner diameter (ft)

    Returns
    -------
    f : Darcy-Weisbach friction factor
    """
    if Re < 1.0:
        return 64.0  # cap for near-zero flow
    if Re < 2000.0:
        return 64.0 / Re   # laminar
    # Turbulent: Serghides (Colebrook-White)
    ed = roughness / D
    A = -2.0 * math.log10(ed / 3.7 + 12.0 / Re)
    B = -2.0 * math.log10(ed / 3.7 + 2.51 / (Re * A))
    C = -2.0 * math.log10(ed / 3.7 + 2.51 / (Re * B))
    f_inv = A - (B - A) ** 2 / (C - 2.0 * B + A)
    f = (1.0 / f_inv) ** 2
    return max(0.005, f)


class PressureLossCalculator:
    """
    Compute pressure loss components along a tubing string for
    multiphase (oil + water + gas) flow.

    Parameters
    ----------
    fluid_props : FluidProperties
        Instance providing mixture PVT properties
    tubing_id : float
        Tubing inner diameter (inches)
    roughness : float
        Absolute pipe wall roughness (ft), default 0.0006 ft
    inclination : float
        Deviation from vertical (degrees). 0 = vertical, 90 = horizontal.
    """

    def __init__(
        self,
        fluid_props,
        tubing_id: float,
        roughness: float = 0.0006,
        inclination: float = 0.0,
    ):
        self.fp = fluid_props
        self.D_in = tubing_id              # inches
        self.D_ft = tubing_id / 12.0       # feet
        self.A = math.pi * self.D_ft ** 2 / 4.0   # ft²
        self.roughness = roughness         # ft
        self.theta = inclination           # degrees from vertical
        self.sin_theta = math.cos(math.radians(inclination))  # sin(from horizontal) = cos(from vertical)

    def _superficial_velocities(self, q_liq_stb: float, P: float, T_F: float):
        """
        Compute superficial velocities for liquid and gas at in-situ conditions.

        Parameters
        ----------
        q_liq_stb : float
            Total liquid surface rate (STB/day)
        P, T_F : float
            Pressure (psia) and temperature (°F)

        Returns
        -------
        v_sl : superficial liquid velocity (ft/s)
        v_sg : superficial gas velocity (ft/s)
        v_m  : mixture velocity (ft/s)
        """
        props = self.fp.liquid_holdup_input(P, T_F)
        ql_ratio = props["ql_res_ratio"]   # RB per STB surface liquid
        qg_ratio = props["qg_res_ratio"]   # RB per STB surface liquid

        # Convert STB/day to ft³/s
        # 1 RB = 5.615 ft³ ; 1 day = 86400 s
        q_L_res = q_liq_stb * ql_ratio * 5.615 / 86400.0   # ft³/s
        q_G_res = q_liq_stb * qg_ratio * 5.615 / 86400.0   # ft³/s

        v_sl = q_L_res / self.A
        v_sg = q_G_res / self.A
        v_m  = v_sl + v_sg
        return v_sl, v_sg, v_m

    # ================================================================== #
    #  BEGGS & BRILL (1973, corrected 1986)                               #
    # ================================================================== #
    def beggs_brill_gradient(
        self, q_liq_stb: float, P: float, T_F: float
    ) -> dict:
        """
        Beggs & Brill multiphase pressure gradient (psi/ft).

        Returns dict with keys:
          gravity_gradient, friction_gradient, accel_gradient, total_gradient
          HL (liquid holdup), rho_mix, flow_regime
        """
        props = self.fp.liquid_holdup_input(P, T_F)
        v_sl, v_sg, v_m = self._superficial_velocities(q_liq_stb, P, T_F)

        rho_L = props["rho_L"]
        rho_g = props["rho_g"]
        mu_L  = props["mu_L"]
        mu_g  = props["mu_g"]
        sigma = props["sigma_L"]
        lambda_L = props["lambda_L"]

        if v_m < 1e-6:
            # No flow: pure hydrostatic
            rho_ns = rho_L * lambda_L + rho_g * (1.0 - lambda_L)
            return {
                "gravity_gradient": rho_ns * self.sin_theta / 144.0,
                "friction_gradient": 0.0,
                "accel_gradient": 0.0,
                "total_gradient": rho_ns * self.sin_theta / 144.0,
                "HL": lambda_L,
                "rho_mix": rho_ns,
                "flow_regime": "No Flow",
            }

        # ---- Flow regime determination --------------------------------
        # Froude number and velocity ratio
        NFr  = v_m ** 2 / (G * self.D_ft)
        NLv  = v_sl * (rho_L / (G * sigma * 6.72e-2)) ** 0.25   # liquid velocity number

        # Transition boundaries (Beggs & Brill, 1986)
        L1 = 316.0 * lambda_L ** 0.302
        L2 = 0.0009252 * lambda_L ** (-2.4684)
        L3 = 0.1 * lambda_L ** (-1.4516)
        L4 = 0.5 * lambda_L ** (-6.738)

        if lambda_L < 0.01 and NFr < L1:
            regime = "Segregated"
        elif lambda_L >= 0.01 and NFr < L2:
            regime = "Segregated"
        elif lambda_L >= 0.01 and NFr > L3 and NFr <= L1:
            regime = "Transition"
        elif lambda_L < 0.4 and NFr >= L1 and NFr <= L4:
            regime = "Intermittent"
        elif lambda_L >= 0.4 and NFr >= L3 and NFr <= L4:
            regime = "Intermittent"
        elif lambda_L < 0.4 and NFr >= L4:
            regime = "Distributed"
        elif lambda_L >= 0.4 and NFr > L4:
            regime = "Distributed"
        else:
            regime = "Segregated"

        # ---- Liquid holdup (horizontal) ----------------------------
        def _HL_horizontal(regime, lambda_L, NFr):
            if regime == "Segregated":
                a, b, c = 0.980, 0.4846, 0.0868
            elif regime == "Intermittent":
                a, b, c = 0.845, 0.5351, 0.0173
            elif regime == "Distributed":
                a, b, c = 1.065, 0.5824, 0.0609
            else:   # Transition – interpolated later
                a, b, c = 0.980, 0.4846, 0.0868
            HL = a * lambda_L ** b / NFr ** c
            return max(lambda_L, min(1.0, HL))

        if regime == "Transition":
            HL_seg = _HL_horizontal("Segregated", lambda_L, NFr)
            HL_int = _HL_horizontal("Intermittent", lambda_L, NFr)
            A_t = (L3 - NFr) / (L3 - L2)
            HL0 = A_t * HL_seg + (1.0 - A_t) * HL_int
        else:
            HL0 = _HL_horizontal(regime, lambda_L, NFr)

        # ---- Inclination correction for upward vertical flow ---------
        # For vertical upward flow, inclination angle from horizontal = 90°
        # beta = 90° for vertical
        if regime == "Segregated":
            e1, e2, e3, f1 = 0.011, -3.768, 3.539, -1.614
        elif regime == "Intermittent":
            e1, e2, e3, f1 = 2.96, 0.305, -0.4473, 0.0978
        else:  # Distributed or Transition
            # No correction for distributed or downhill
            HL = HL0
            e1 = e2 = e3 = f1 = 0.0

        if e1 != 0.0:
            NLv_loc = v_sl * (rho_L / (G * sigma * 6.72e-2)) ** 0.25
            C = max(0.0, (1.0 - lambda_L) * math.log(
                abs(e1) * lambda_L ** e2 * NLv_loc ** e3 * NFr ** f1 + 1e-10
            ))
            psi = 1.0 + C * (math.sin(1.8 * math.radians(90.0))
                              - (1.0/3.0) * math.sin(1.8 * math.radians(90.0)) ** 3)
            HL = HL0 * psi
        else:
            HL = HL0

        HL = max(lambda_L, min(1.0, HL))

        # ---- Mixture properties with holdup --------------------------
        rho_mix = rho_L * HL + rho_g * (1.0 - HL)
        rho_ns  = rho_L * lambda_L + rho_g * (1.0 - lambda_L)

        # ---- Friction factor ------------------------------------------
        mu_m  = mu_L ** lambda_L * mu_g ** (1.0 - lambda_L)
        Re_m  = rho_ns * v_m * self.D_ft / (mu_m * 6.72e-4)   # cp→lb/(ft·s)
        fn    = moody_friction_factor(Re_m, self.roughness, self.D_ft)

        # Holdup correction to friction factor (Beggs & Brill)
        if lambda_L > 0.0 and HL > 0.0:
            y = lambda_L / HL ** 2
            if 1.0 < y < np.inf:
                ln_y = math.log(y)
                s_num = ln_y
                s_den = (-0.0523 + 3.182*ln_y - 0.8725*ln_y**2 + 0.01853*ln_y**4)
                s = s_num / max(abs(s_den), 0.001)
                s = max(-2.0, min(2.0, s))
                f_tp = fn * math.exp(s)
            else:
                f_tp = fn
        else:
            f_tp = fn

        # ---- Pressure gradients (psi/ft) -----------------------------
        # Gravity (hydrostatic)
        dP_grav = rho_mix * self.sin_theta / 144.0

        # Friction  (Darcy-Weisbach)
        dP_fric = f_tp * rho_ns * v_m ** 2 / (2.0 * G_C * self.D_ft * 144.0)

        # Acceleration (kinetic energy)
        # dP_acc = rho_mix × vm × dvm/dL ≈ 0 for incompressible, small for gas
        Ek = rho_ns * v_m * v_sg / (G_C * P * 144.0)  # dimensionless
        if Ek >= 1.0:
            Ek = 0.99
        dP_acc = dP_grav + dP_fric   # placeholder: total/(1-Ek)  later

        total_grad = (dP_grav + dP_fric) / max(1.0 - Ek, 0.01)

        return {
            "gravity_gradient":  dP_grav,
            "friction_gradient": dP_fric,
            "accel_gradient":    total_grad - dP_grav - dP_fric,
            "total_gradient":    total_grad,
            "HL": HL,
            "rho_mix": rho_mix,
            "flow_regime": regime,
        }

    # ================================================================== #
    #  HAGEDORN & BROWN (1965) with Griffith-Wallis bubble correction      #
    # ================================================================== #
    def hagedorn_brown_gradient(
        self, q_liq_stb: float, P: float, T_F: float
    ) -> dict:
        """
        Hagedorn & Brown (1965) correlation.
        Corrected with Griffith-Wallis bubble-flow check.

        Returns same dict structure as beggs_brill_gradient.
        """
        props = self.fp.liquid_holdup_input(P, T_F)
        v_sl, v_sg, v_m = self._superficial_velocities(q_liq_stb, P, T_F)

        rho_L = props["rho_L"]
        rho_g = props["rho_g"]
        mu_L  = props["mu_L"]
        mu_g  = props["mu_g"]
        sigma = props["sigma_L"]
        lambda_L = props["lambda_L"]

        if v_m < 1e-6:
            rho_ns = rho_L * lambda_L + rho_g * (1.0 - lambda_L)
            return {
                "gravity_gradient": rho_ns / 144.0,
                "friction_gradient": 0.0,
                "accel_gradient": 0.0,
                "total_gradient": rho_ns / 144.0,
                "HL": lambda_L,
                "rho_mix": rho_ns,
                "flow_regime": "No Flow",
            }

        # ---- Griffith-Wallis bubble flow check -----------------------
        # If flow is in bubble regime, use simplified approach
        vb = 0.8   # bubble rise velocity (ft/s) – approximate
        if v_sg < 0.25 * v_m and v_sg < vb:
            # Bubble flow: simple mixture approach
            HL_bub = 1.0 - v_sg / (0.8 + v_m)
            HL_bub = max(lambda_L, min(1.0, HL_bub))
            regime = "Bubble"
        else:
            HL_bub = None
            regime = "Slug/Mist"

        if HL_bub is not None:
            HL = HL_bub
        else:
            # ---- CNL correlation dimensionless numbers ---------------
            # Liquid velocity number
            NLv = v_sl * (rho_L / (G * sigma * 6.72e-2)) ** 0.25
            # Gas velocity number
            NGv = v_sg * (rho_L / (G * sigma * 6.72e-2)) ** 0.25
            # Pipe diameter number
            Nd  = self.D_ft * (rho_L * G / (sigma * 6.72e-2)) ** 0.5
            # Liquid viscosity number
            NL  = mu_L * (G / (rho_L * (sigma * 6.72e-2) ** 3)) ** 0.25

            # ---- CNL/Pr holdup correlation (Hagedorn & Brown tables → polynomial fit)
            # Using the commonly cited polynomial approximations
            # CNL correlation: HL/psi
            CNL_coeff = 0.002  # liquid viscosity factor (approx)

            # Pseudo-reduced pressure ratio
            psi_r = (NLv / NGv ** 0.575) * (14.7 / P) ** 0.1 * (CNL_coeff / Nd)
            # Clamp
            psi_r = max(0.01, min(100.0, psi_r))

            # HL polynomial fit (from H&B chart regression)
            # log(HL) vs log(NLv*Pvac^0.1/(NGv^0.575 * Nd * CNL))
            lp = math.log10(psi_r)
            HL = 10 ** (0.816 * lp - 0.0694 * lp**2 + 0.0028 * lp**3)
            HL = max(lambda_L, min(1.0, HL))

        # ---- Mixture density -----------------------------------------
        rho_mix = rho_L * HL + rho_g * (1.0 - HL)
        rho_ns  = rho_L * lambda_L + rho_g * (1.0 - lambda_L)

        # ---- Friction factor (using mixture Reynolds) ----------------
        mu_m  = mu_L ** HL * mu_g ** (1.0 - HL)
        Re_m  = rho_mix * v_m * self.D_ft / (mu_m * 6.72e-4)
        f_tp  = moody_friction_factor(Re_m, self.roughness, self.D_ft)

        # ---- Pressure gradients -------------------------------------
        dP_grav = rho_mix * self.sin_theta / 144.0
        dP_fric = f_tp * rho_mix * v_m ** 2 / (2.0 * G_C * self.D_ft * 144.0)

        # Acceleration (Poettmann & Carpenter term)
        Ek = rho_mix * v_m * v_sg / (G_C * P * 144.0)
        if Ek >= 1.0:
            Ek = 0.99
        total_grad = (dP_grav + dP_fric) / max(1.0 - Ek, 0.01)

        return {
            "gravity_gradient":  dP_grav,
            "friction_gradient": dP_fric,
            "accel_gradient":    total_grad - dP_grav - dP_fric,
            "total_gradient":    total_grad,
            "HL": HL,
            "rho_mix": rho_mix,
            "flow_regime": regime,
        }


## 📉 Module 3: Inflow Performance Relationship (IPR)
Models fluid inflow from the reservoir into the wellbore across the perforations.
* **Vogel (1968):** Two-phase solution-gas drive reservoirs below bubble point.
* **Fetkovitch (1973):** Back-pressure equation ($q = C(P_r^2 - P_{wf}^2)^n$).
* **Jones, Blount & Glaze (1976):** High-velocity turbulent flow ($P_r - P_{wf} = A q + B q^2$).
* **Standing (1970):** Modified Vogel accounting for Flow Efficiency (FE / Skin).
* **Darcy:** Linear single-phase liquid flow above bubble point.


In [ ]:
from typing import Tuple


class IPRCalculator:
    """
    Compute IPR curves for a reservoir.

    Parameters
    ----------
    reservoir_pressure : float
        Static reservoir pressure Pr (psia)
    productivity_index : float
        Productivity index J (STB/day/psi)  – used by Linear PI & Standing
    bubble_point : float
        Bubble-point pressure Pb (psia)
    reservoir_temp : float
        Reservoir temperature (°F) – used for Fetkovitch
    jones_a : float, optional
        Jones turbulence coefficient A (psi²/cp / (STB/day)²) – Jones model only
    jones_b : float, optional
        Jones Darcy coefficient B (psi²/cp / (STB/day)) – Jones model only
    fetkovitch_n : float, optional
        Fetkovitch flow exponent n (0.5 – 1.0, default 1.0 = Darcy)
    completion_efficiency : float, optional
        Standing completion efficiency FE (0 – 1.5, default 1.0)
    """

    def __init__(
        self,
        reservoir_pressure: float,
        productivity_index: float,
        bubble_point: float,
        reservoir_temp: float = 180.0,
        jones_a: float = 0.0,
        jones_b: float = None,
        fetkovitch_n: float = 1.0,
        completion_efficiency: float = 1.0,
    ):
        self.Pr = reservoir_pressure
        self.PI = productivity_index
        self.Pb = bubble_point
        self.T = reservoir_temp
        self.jones_a = jones_a
        self.jones_b = jones_b if jones_b is not None else productivity_index
        self.n = max(0.5, min(1.0, fetkovitch_n))
        self.FE = completion_efficiency

        # AOF (Absolute Open Flow) at Pwf = 0 for the selected model
        self._aof_cache = {}

    # ------------------------------------------------------------------ #
    # 1. LINEAR PI MODEL                                                   #
    # ------------------------------------------------------------------ #
    def _linear_pi(self, Pwf: float) -> float:
        """q = PI × (Pr − Pwf). Valid above and below bubble point."""
        if Pwf >= self.Pr:
            return 0.0
        return max(0.0, self.PI * (self.Pr - Pwf))

    # ------------------------------------------------------------------ #
    # 2. VOGEL MODEL (1968)                                                #
    # ------------------------------------------------------------------ #
    def _vogel(self, Pwf: float) -> float:
        """
        Vogel (1968) IPR for solution-gas drive reservoirs.
        qo_max estimated from PI at bubble-point or above.
        """
        if Pwf >= self.Pr:
            return 0.0
        qo_max = self._vogel_aof()
        x = Pwf / self.Pr
        q = qo_max * (1.0 - 0.2 * x - 0.8 * x ** 2)
        return max(0.0, q)

    def _vogel_aof(self) -> float:
        """Max AOF for Vogel at Pwf=0."""
        if "vogel" in self._aof_cache:
            return self._aof_cache["vogel"]
        # Use PI to anchor: at Pwf=Pb, q_Pb = PI*(Pr-Pb) if Pr>Pb, else linear
        if self.Pr > self.Pb > 0:
            q_at_pb = self.PI * (self.Pr - self.Pb)
            # Vogel fraction at Pwf=Pb
            x = self.Pb / self.Pr
            vogel_frac = 1.0 - 0.2 * x - 0.8 * x ** 2
            qo_max = q_at_pb / vogel_frac if vogel_frac > 0 else self.PI * self.Pr
        else:
            # No gas: pure linear
            qo_max = self.PI * self.Pr
        self._aof_cache["vogel"] = qo_max
        return qo_max

    # ------------------------------------------------------------------ #
    # 3. FETKOVITCH MODEL (1973)                                           #
    # ------------------------------------------------------------------ #
    def _fetkovitch(self, Pwf: float) -> float:
        """
        Fetkovitch (1973): q = C × (Pr² − Pwf²)^n
        C is derived from PI at pseudo-steady-state.
        n = flow exponent (0.5 = fully turbulent, 1.0 = Darcy)
        """
        if Pwf >= self.Pr:
            return 0.0
        # Derive C from PI: at Pwf near Pr, dq/dPwf ≈ PI
        # PI ≈ C × n × (Pr²-Pwf²)^(n-1) × 2Pwf → at Pwf→Pr: indeterminate
        # Use: q(Pwf=0) = PI*Pr from linear → C = PI*Pr / Pr^(2n)
        C = self.PI * self.Pr / (self.Pr ** (2 * self.n))
        dp2 = max(0.0, self.Pr ** 2 - Pwf ** 2)
        q = C * dp2 ** self.n
        return max(0.0, q)

    # ------------------------------------------------------------------ #
    # 4. STANDING MODEL (modified Vogel with Efficiency, 1970)             #
    # ------------------------------------------------------------------ #
    def _standing(self, Pwf: float) -> float:
        """
        Standing (1970) modified Vogel with completion efficiency FE.
        Vogel equation with FE factor on the linear + quadratic terms.
        """
        if Pwf >= self.Pr:
            return 0.0
        qo_max = self._standing_aof()
        x = Pwf / self.Pr
        q = qo_max * (1.0 - 0.2 * (x / self.FE) - 0.8 * (x / self.FE) ** 2)
        return max(0.0, q)

    def _standing_aof(self) -> float:
        """AOF for Standing model."""
        if "standing" in self._aof_cache:
            return self._aof_cache["standing"]
        # Same anchoring as Vogel but with FE
        if self.Pr > self.Pb > 0:
            q_at_pb = self.PI * (self.Pr - self.Pb)
            x = self.Pb / self.Pr
            standing_frac = 1.0 - 0.2 * (x / self.FE) - 0.8 * (x / self.FE) ** 2
            qo_max = q_at_pb / max(standing_frac, 0.01)
        else:
            qo_max = self.PI * self.Pr * self.FE
        self._aof_cache["standing"] = qo_max
        return qo_max

    # ------------------------------------------------------------------ #
    # 5. JONES COMPOSITE MODEL (Jones, Blount & Glaze, 1976)               #
    # ------------------------------------------------------------------ #
    def _jones(self, Pwf: float) -> float:
        """
        Jones composite model: Pr − Pwf = A×q² + B×q
        A = non-Darcy (turbulence) coefficient [psi/(STB/day)²]
        B = Darcy coefficient [psi/(STB/day)]

        Solved by quadratic formula: q = (−B + sqrt(B²+4A×ΔP)) / (2A)
        """
        if Pwf >= self.Pr:
            return 0.0
        dP = self.Pr - Pwf
        A = self.jones_a
        B = self.jones_b if self.jones_b else 1.0 / self.PI
        if abs(A) < 1e-12:
            # Pure Darcy (B only)
            return max(0.0, dP / B)
        discriminant = B ** 2 + 4.0 * A * dP
        q = (-B + discriminant ** 0.5) / (2.0 * A)
        return max(0.0, q)

    # ------------------------------------------------------------------ #
    # PUBLIC API                                                           #
    # ------------------------------------------------------------------ #
    def compute_rate(self, Pwf: float, model: str = "vogel") -> float:
        """
        Compute flow rate for a given Pwf.

        Parameters
        ----------
        Pwf   : bottomhole flowing pressure (psia)
        model : one of 'linear_pi', 'vogel', 'fetkovitch', 'standing', 'jones'

        Returns
        -------
        q : flow rate (STB/day)
        """
        model = model.lower().replace(" ", "_")
        dispatch = {
            "linear_pi": self._linear_pi,
            "linear pi": self._linear_pi,
            "vogel": self._vogel,
            "fetkovitch": self._fetkovitch,
            "standing": self._standing,
            "jones": self._jones,
        }
        fn = dispatch.get(model)
        if fn is None:
            raise ValueError(f"Unknown IPR model: '{model}'. "
                             f"Choose from: {list(dispatch.keys())}")
        return fn(Pwf)

    def ipr_curve(
        self, model: str = "vogel", n_points: int = 100
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate a full IPR curve (q, Pwf arrays).

        Returns
        -------
        q_array    : np.ndarray of flow rates (STB/day), high→low
        pwf_array  : np.ndarray of BHFPs (psia), high→low
        """
        pwf_array = np.linspace(self.Pr, 0.0, n_points)
        q_array = np.array([self.compute_rate(p, model) for p in pwf_array])
        return q_array, pwf_array

    def aof(self, model: str = "vogel") -> float:
        """Absolute Open Flow at Pwf = 0 for the given model."""
        return self.compute_rate(0.0, model)

    def all_models_comparison(
        self, n_points: int = 100
    ) -> dict:
        """
        Return IPR curves for all 5 models.
        Returns dict: {model_name: (q_array, pwf_array)}
        """
        models = ["linear_pi", "vogel", "fetkovitch", "standing", "jones"]
        results = {}
        for m in models:
            results[m] = self.ipr_curve(m, n_points)
        return results


## 📈 Module 4: Vertical Lift Performance (VLP) & Tubing Sizing
Integrates pressure gradients from the wellhead down to the perforations (mid-perforation depth) to construct the outflow VLP curve.
Includes standard API tubing sizes and multi-tubing evaluation capabilities.


In [ ]:
# Standard API tubing sizes: {name: ID in inches}
STANDARD_TUBING_SIZES = {
    "2-3/8\" (1.995\" ID)": 1.995,
    "2-7/8\" (2.441\" ID)": 2.441,
    "3-1/2\" (2.992\" ID)": 2.992,
    "4\" (3.476\" ID)": 3.476,
    "4-1/2\" (3.958\" ID)": 3.958,
}


class VLPCalculator:
    """
    Compute the VLP (outflow) curve for a well.

    Parameters
    ----------
    fluid_props : FluidProperties
        PVT fluid properties object
    tvd : float
        True vertical depth to mid-perforation (ft)
    md : float
        Measured depth to mid-perforation (ft)
    tubing_id : float
        Tubing inner diameter (inches)
    wellhead_pressure : float
        Flowing wellhead (tubing head) pressure (psia), default 100
    reservoir_temp : float
        Temperature at perforations (°F)
    surface_temp : float
        Temperature at wellhead (°F), default 75
    roughness : float
        Pipe wall roughness (ft), default 0.0006
    n_segments : int
        Number of integration segments, default 50
    """

    def __init__(
        self,
        fluid_props,
        tvd: float,
        md: float,
        tubing_id: float,
        wellhead_pressure: float = 100.0,
        reservoir_temp: float = 180.0,
        surface_temp: float = 75.0,
        roughness: float = 0.0006,
        n_segments: int = 50,
    ):
        self.fp = fluid_props
        self.tvd = tvd
        self.md = md
        self.tubing_id = tubing_id
        self.whp = wellhead_pressure
        self.T_res = reservoir_temp
        self.T_surf = surface_temp
        self.roughness = roughness
        self.n_seg = n_segments

        # Average inclination (degrees from vertical)
        # For deviated wells, sin(avg inclination) corrects hydrostatic
        self.avg_inclination = self._avg_inclination()

        self.plc = PressureLossCalculator(
            fluid_props=self.fp,
            tubing_id=self.tubing_id,
            roughness=self.roughness,
            inclination=self.avg_inclination,
        )

    def _avg_inclination(self) -> float:
        """Compute average deviation angle (degrees from vertical)."""
        if self.md <= self.tvd or self.md < 1.0:
            return 0.0
        # Simple: average inclination = acos(TVD/MD)
        ratio = min(1.0, self.tvd / self.md)
        return math.degrees(math.acos(ratio))

    def _temperature_at_depth(self, depth_ft: float) -> float:
        """
        Linear temperature profile from wellhead (surface_temp) to TVD (reservoir_temp).
        depth_ft is measured from surface (0 = surface, tvd = reservoir).
        """
        if self.tvd <= 0:
            return self.T_res
        frac = depth_ft / self.tvd
        return self.T_surf + frac * (self.T_res - self.T_surf)

    def compute_fbhp(
        self,
        q_liq: float,
        correlation: str = "beggs_brill",
    ) -> dict:
        """
        Compute FBHP at perforations for a given surface liquid rate.

        Integrates pressure from wellhead → perforations in n_seg steps.
        Returns dict with:
          fbhp        : flowing BHP at perforations (psia)
          dP_gravity  : total hydrostatic pressure loss (psi)
          dP_friction : total friction pressure loss (psi)
          dP_accel    : total acceleration pressure loss (psi)
          dP_total    : total pressure loss (psi)
          profile     : list of (depth, pressure) tuples for plotting
        """
        # Build depth array from wellhead (0) to TVD
        depths = np.linspace(0.0, self.tvd, self.n_seg + 1)
        dz = self.tvd / self.n_seg   # depth increment per segment (ft)

        P_current = self.whp   # start at wellhead
        dP_grav_total = 0.0
        dP_fric_total = 0.0
        dP_acc_total  = 0.0
        profile = [(0.0, P_current)]

        for i in range(self.n_seg):
            depth_mid = (depths[i] + depths[i + 1]) / 2.0
            T_mid = self._temperature_at_depth(depth_mid)
            P_mid = max(14.7, P_current)

            if correlation == "beggs_brill":
                grad = self.plc.beggs_brill_gradient(q_liq, P_mid, T_mid)
            elif correlation == "hagedorn_brown":
                grad = self.plc.hagedorn_brown_gradient(q_liq, P_mid, T_mid)
            else:
                raise ValueError(f"Unknown correlation: '{correlation}'")

            dP_g = grad["gravity_gradient"]  * dz
            dP_f = grad["friction_gradient"] * dz
            dP_a = grad["accel_gradient"]    * dz

            P_current += dP_g + dP_f + dP_a
            P_current = max(14.7, min(50000.0, P_current))

            dP_grav_total += dP_g
            dP_fric_total += dP_f
            dP_acc_total  += dP_a

            profile.append((depths[i + 1], P_current))

        fbhp = P_current
        dP_total = fbhp - self.whp

        return {
            "fbhp": fbhp,
            "dP_gravity":  dP_grav_total,
            "dP_friction": dP_fric_total,
            "dP_accel":    dP_acc_total,
            "dP_total":    dP_total,
            "profile": profile,
        }

    def vlp_curve(
        self,
        q_max: float,
        correlation: str = "beggs_brill",
        n_points: int = 60,
    ) -> tuple:
        """
        Generate the full VLP curve (q → FBHP).

        Parameters
        ----------
        q_max        : maximum flow rate to evaluate (STB/day)
        correlation  : 'beggs_brill' or 'hagedorn_brown'
        n_points     : number of points

        Returns
        -------
        (q_array, fbhp_array) as numpy arrays
        """
        # Use a log-scale so we capture the curve knee well
        q_min = max(1.0, q_max * 0.005)
        q_vals = np.concatenate([
            np.linspace(0.0, q_min, 3),
            np.geomspace(q_min, q_max, n_points - 3)
        ])
        q_vals = np.unique(q_vals)

        fbhp_vals = []
        for q in q_vals:
            if q <= 0:
                # At q=0: FBHP = hydrostatic head only
                res = self.compute_fbhp(1e-3, correlation)
                # For zero flow, friction = 0, only gravity
                fbhp_vals.append(self.whp + res["dP_gravity"])
            else:
                res = self.compute_fbhp(q, correlation)
                fbhp_vals.append(res["fbhp"])

        return np.array(q_vals), np.array(fbhp_vals)

    def multi_tubing_vlp(
        self,
        tubing_sizes: dict,
        q_max: float,
        correlation: str = "beggs_brill",
        n_points: int = 60,
    ) -> dict:
        """
        Compute VLP curves for multiple tubing sizes.

        Parameters
        ----------
        tubing_sizes : dict mapping name → inner diameter (inches)
        q_max        : maximum surface rate (STB/day)
        correlation  : 'beggs_brill' or 'hagedorn_brown'

        Returns
        -------
        dict: {tubing_name: (q_array, fbhp_array)}
        """
        results = {}
        for name, id_in in tubing_sizes.items():
            calc = VLPCalculator(
                fluid_props=self.fp,
                tvd=self.tvd,
                md=self.md,
                tubing_id=id_in,
                wellhead_pressure=self.whp,
                reservoir_temp=self.T_res,
                surface_temp=self.T_surf,
                roughness=self.roughness,
                n_segments=self.n_seg,
            )
            q_arr, fbhp_arr = calc.vlp_curve(q_max, correlation, n_points)
            results[name] = (q_arr, fbhp_arr)
        return results

    def pressure_loss_breakdown(
        self,
        q_liq: float,
        correlation: str = "beggs_brill",
    ) -> dict:
        """
        Detailed pressure loss breakdown at a specific flow rate.
        Returns gravity, friction, acceleration, and total losses (psi).
        """
        res = self.compute_fbhp(q_liq, correlation)
        return {
            "Hydrostatic (Gravity)": res["dP_gravity"],
            "Friction":              res["dP_friction"],
            "Acceleration":          res["dP_accel"],
            "Total ΔP":              res["dP_total"],
            "FBHP":                  res["fbhp"],
        }


## 🎯 Module 5: Nodal Analysis Solver & System Orchestrator
Combines IPR (inflow) and VLP (outflow) at the wellbore node.
Uses a robust bisection algorithm to find the exact zero-crossing where:
$$\text{FBHP}_{\text{IPR}}(q) = \text{FBHP}_{\text{VLP}}(q)$$


In [ ]:
class NodalAnalysis:
    """
    Nodal analysis solver combining IPR and VLP.

    Parameters
    ----------
    ipr_calc : IPRCalculator
    vlp_calc : VLPCalculator
    """

    def __init__(self, ipr_calc: IPRCalculator, vlp_calc: VLPCalculator):
        self.ipr = ipr_calc
        self.vlp = vlp_calc

    def operating_point(
        self,
        ipr_model: str = "vogel",
        vlp_correlation: str = "beggs_brill",
        n_points: int = 80,
        tol: float = 1.0,   # psi tolerance
    ) -> dict:
        """
        Find the operating point (intersection of IPR and VLP curves).

        Algorithm:
          1. Compute IPR at n_points from 0 → AOF
          2. Compute VLP at the same flow rates
          3. Find sign change in (IPR_Pwf - VLP_Pwf)
          4. Bisect within that bracket for precision

        Returns
        -------
        dict with:
          q     : operating flow rate (STB/day)
          Pwf   : operating FBHP (psia)
          aof   : Absolute Open Flow (STB/day) at Pwf=0
          found : bool — True if intersection was found
          message : descriptive string
        """
        aof = self.ipr.aof(ipr_model)
        if aof <= 0:
            return {
                "q": 0.0, "Pwf": self.ipr.Pr,
                "aof": 0.0, "found": False,
                "message": "No production: AOF = 0."
            }

        # Build arrays
        q_arr = np.linspace(0.0, aof * 1.05, n_points)
        ipr_pwf = np.array([self.ipr.compute_rate(0.0, ipr_model)] * n_points)  # placeholder
        # IPR: q → Pwf (invert the IPR curve)
        # Build q→Pwf for IPR by interpolation
        q_ipr, pwf_ipr = self.ipr.ipr_curve(ipr_model, n_points * 2)
        # Ensure monotone (q_ipr should be ascending)
        sort_idx = np.argsort(q_ipr)
        q_ipr_s = q_ipr[sort_idx]
        pwf_ipr_s = pwf_ipr[sort_idx]

        # Remove duplicates
        _, uniq = np.unique(q_ipr_s, return_index=True)
        q_ipr_s = q_ipr_s[uniq]
        pwf_ipr_s = pwf_ipr_s[uniq]

        if len(q_ipr_s) < 2:
            return {
                "q": 0.0, "Pwf": self.ipr.Pr,
                "aof": aof, "found": False,
                "message": "IPR curve degenerate."
            }

        # Interpolator for IPR Pwf given q
        ipr_interp = interp1d(
            q_ipr_s, pwf_ipr_s,
            kind="linear",
            bounds_error=False,
            fill_value=(pwf_ipr_s[0], pwf_ipr_s[-1])
        )

        # VLP: compute FBHP at same q points
        q_eval = np.linspace(0.5, aof, n_points)
        vlp_fbhp = np.array([
            self.vlp.compute_fbhp(q, vlp_correlation)["fbhp"]
            for q in q_eval
        ])
        ipr_fbhp = ipr_interp(q_eval)

        # Difference: IPR_Pwf - VLP_Pwf
        diff = ipr_fbhp - vlp_fbhp

        # Find zero crossing
        sign_changes = np.where(np.diff(np.sign(diff)))[0]

        if len(sign_changes) == 0:
            # No intersection found — check which side
            if diff[-1] > 0:
                # VLP always below IPR → well flows at AOF (limited by wellbore)
                q_op = aof
                pwf_op = float(ipr_interp(aof))
                msg = "VLP below IPR for all rates: system may be flow-limited."
            else:
                # VLP always above IPR → well cannot produce
                q_op = 0.0
                pwf_op = float(self.ipr.Pr)
                msg = "VLP above IPR for all rates: well cannot produce at these conditions."
            return {
                "q": q_op, "Pwf": pwf_op,
                "aof": aof, "found": False,
                "message": msg,
            }

        # Use the first (lowest-rate) intersection
        idx = sign_changes[0]
        q_lo, q_hi = q_eval[idx], q_eval[idx + 1]

        # Bisection refinement
        for _ in range(60):
            q_mid = (q_lo + q_hi) / 2.0
            diff_mid = float(ipr_interp(q_mid)) - self.vlp.compute_fbhp(q_mid, vlp_correlation)["fbhp"]
            diff_lo  = float(ipr_interp(q_lo))  - self.vlp.compute_fbhp(q_lo,  vlp_correlation)["fbhp"]
            if abs(q_hi - q_lo) < 0.1:
                break
            if diff_lo * diff_mid < 0:
                q_hi = q_mid
            else:
                q_lo = q_mid

        q_op   = (q_lo + q_hi) / 2.0
        pwf_op = self.vlp.compute_fbhp(q_op, vlp_correlation)["fbhp"]

        return {
            "q":    round(q_op, 1),
            "Pwf":  round(pwf_op, 1),
            "aof":  round(aof, 1),
            "found": True,
            "message": f"Operating point: q = {q_op:.0f} STB/day, Pwf = {pwf_op:.0f} psia",
        }

    def full_analysis(
        self,
        ipr_model: str = "vogel",
        vlp_correlation: str = "beggs_brill",
        n_curve_points: int = 80,
    ) -> dict:
        """
        Run complete nodal analysis and return all data for plotting.

        Returns
        -------
        dict with:
          ipr_curves       : dict {model: (q, pwf)} for all models
          vlp_q, vlp_fbhp  : VLP curve arrays
          operating_point  : {q, Pwf, aof, found, message}
          pressure_losses  : breakdown dict at operating point
          reservoir_info   : summary dict
        """
        aof_primary = self.ipr.aof(ipr_model)
        q_max = max(aof_primary * 1.1, 100.0)

        # ---- IPR curves (all models) ---------------------------------
        ipr_curves = self.ipr.all_models_comparison(n_curve_points)

        # ---- VLP curve ----------------------------------------------
        vlp_q, vlp_fbhp = self.vlp.vlp_curve(q_max, vlp_correlation, n_curve_points)

        # ---- Operating point ----------------------------------------
        op = self.operating_point(ipr_model, vlp_correlation)

        # ---- Pressure loss breakdown at operating point --------------
        if op["q"] > 0:
            pl = self.vlp.pressure_loss_breakdown(op["q"], vlp_correlation)
        else:
            pl = {
                "Hydrostatic (Gravity)": 0.0,
                "Friction": 0.0,
                "Acceleration": 0.0,
                "Total ΔP": 0.0,
                "FBHP": self.ipr.Pr,
            }

        # ---- Summary info -------------------------------------------
        reservoir_info = {
            "Reservoir Pressure (psia)":    self.ipr.Pr,
            "Bubble-Point Pressure (psia)": self.ipr.Pb,
            "Productivity Index (STB/d/psi)": self.ipr.PI,
            "AOF (STB/day)":               aof_primary,
            "Operating Rate (STB/day)":     op["q"],
            "Operating FBHP (psia)":        op["Pwf"],
            "Drawdown (psia)":              round(self.ipr.Pr - op["Pwf"], 1),
            "Wellhead Pressure (psia)":     self.vlp.whp,
            "TVD (ft)":                     self.vlp.tvd,
            "Tubing ID (in)":               self.vlp.tubing_id,
        }

        return {
            "ipr_curves":      ipr_curves,
            "vlp_q":           vlp_q,
            "vlp_fbhp":        vlp_fbhp,
            "operating_point": op,
            "pressure_losses": pl,
            "reservoir_info":  reservoir_info,
        }

    def sensitivity_analysis(
        self,
        parameter: str,
        values: list,
        ipr_model: str = "vogel",
        vlp_correlation: str = "beggs_brill",
    ) -> list:
        """
        Run nodal analysis for a range of parameter values.
        Useful for sensitivity plots (e.g., WHP sensitivity).

        Parameters
        ----------
        parameter : 'whp', 'water_cut', 'gor', 'tubing_id'
        values    : list of values to test
        ...

        Returns
        -------
        list of {param_value, q, Pwf}
        """
        results = []
        original_whp = self.vlp.whp
        original_wc  = self.vlp.fp.wc
        original_gor = self.vlp.fp.gor
        original_tid = self.vlp.tubing_id

        for v in values:
            try:
                if parameter == "whp":
                    self.vlp.whp = v
                elif parameter == "water_cut":
                    self.vlp.fp.wc = v
                    # Recompute bubble-point
                    self.vlp.fp.bubble_point = self.vlp.fp._bubble_point_standing(
                        self.vlp.fp.T_res, self.vlp.fp.gor
                    )
                elif parameter == "gor":
                    self.vlp.fp.gor = v
                    self.vlp.fp.bubble_point = self.vlp.fp._bubble_point_standing(
                        self.vlp.fp.T_res, v
                    )
                elif parameter == "tubing_id":
                    self.vlp.tubing_id = v
                    self.vlp.D_ft = v / 12.0
                    self.vlp.plc.D_ft = v / 12.0
                    self.vlp.plc.D_in = v
                    self.vlp.plc.A = math.pi * (v / 12.0) ** 2 / 4.0

                op = self.operating_point(ipr_model, vlp_correlation)
                results.append({
                    "param_value": v,
                    "q": op["q"],
                    "Pwf": op["Pwf"],
                    "aof": op["aof"],
                })
            except Exception as e:
                results.append({"param_value": v, "q": 0.0, "Pwf": 0.0, "aof": 0.0})

        # Restore original values
        self.vlp.whp = original_whp
        self.vlp.fp.wc = original_wc
        self.vlp.fp.gor = original_gor
        self.vlp.tubing_id = original_tid
        if hasattr(self.vlp, 'plc'):
            self.vlp.plc.D_ft = original_tid / 12.0
            self.vlp.plc.D_in = original_tid
            self.vlp.plc.A = math.pi * (original_tid / 12.0) ** 2 / 4.0

        return results


## 🚀 Module 6: Run Nodal Analysis & Interactive Visualization
In the code cell below, you can modify any reservoir, tubing, or PVT parameter and execute the analysis. The engine will solve for the operating point and display an interactive **Plotly chart** directly in your Colab output!


In [ ]:
# =====================================================================
# STEP 2: Configure Well Parameters & Run Simulation
# =====================================================================
# ── 1. Reservoir & PVT Parameters ──
reservoir_pressure = 4200.0   # Pr (psia)
reservoir_temp = 180.0        # T_res (°F)
productivity_index = 2.5      # PI (STB/day/psi)
api_gravity = 35.0            # °API
gas_sg = 0.65                 # Gas specific gravity (air = 1.0)
water_sg = 1.07               # Water specific gravity
gor = 500.0                   # Producing GOR (scf/STB)
water_cut = 0.20              # Water cut (0.20 = 20%)

# ── 2. Wellbore & Tubing Parameters ──
tvd = 8000.0                  # True Vertical Depth (ft)
md = 9000.0                   # Measured Depth (ft)
tubing_id = 2.992             # Tubing Inner Diameter (in) -> 3-1/2" OD
roughness = 0.0006            # Pipe wall roughness (ft)
whp = 250.0                   # Wellhead Pressure (psia)
surface_temp = 100.0          # Wellhead Temperature (°F)

# ── 3. Model Selection ──
ipr_model = "vogel"           # Options: 'vogel', 'fetkovitch', 'jones', 'standing', 'darcy'
vlp_correlation = "beggs_brill" # Options: 'beggs_brill', 'hagedorn_brown'

# =====================================================================
# EXECUTE SOLVER
# =====================================================================
print("⚙️ Running Nodal Analysis Solver...")
fp = FluidProperties(api_gravity, gas_sg, water_sg, gor, water_cut, reservoir_temp)
ipr_calc = IPRCalculator(reservoir_pressure, productivity_index, fp.bubble_point, reservoir_temp)
vlp_calc = VLPCalculator(fp, tvd, md, tubing_id, whp, reservoir_temp, surface_temp, roughness)

solver = NodalAnalysis(ipr_calc, vlp_calc)
results = solver.full_analysis(ipr_model, vlp_correlation)
op = results["operating_point"]

# ── Print Summary Table ──
print("\n" + "="*60)
print(f"📊 NODAL ANALYSIS RESULTS ({op['message']})")
print("="*60)
for k, v in results["reservoir_info"].items():
    print(f"  • {k:<32} : {v}")
print("="*60)

# ── Plot Interactive IPR / VLP Chart ──
q_ipr, pwf_ipr = ipr_calc.ipr_curve(ipr_model, n_points=100)
vlp_q = results["vlp_q"]
vlp_fbhp = results["vlp_fbhp"]

fig = go.Figure()

# IPR Curve
fig.add_trace(go.Scatter(
    x=q_ipr, y=pwf_ipr, mode='lines', name=f'IPR ({ipr_model.upper()})',
    line=dict(color='#00E5FF', width=3.5)
))

# VLP Curve
fig.add_trace(go.Scatter(
    x=vlp_q, y=vlp_fbhp, mode='lines', name=f'VLP ({vlp_correlation.replace("_", " ").title()})',
    line=dict(color='#FF5252', width=3.5)
))

# Operating Point
if op["found"]:
    fig.add_trace(go.Scatter(
        x=[op["q"]], y=[op["Pwf"]], mode='markers+text', name='Operating Point',
        marker=dict(color='#00E676', size=14, line=dict(color='white', width=2)),
        text=[f"  q={op['q']:.0f} STB/d<br>  Pwf={op['Pwf']:.0f} psia"],
        textposition="top right",
        textfont=dict(color='#00E676', size=12, family="monospace")
    ))

fig.update_layout(
    title=f"<b>Nodal Analysis — Inflow vs. Outflow Intersection</b><br><sup>Tubing ID: {tubing_id} in | WHP: {whp} psia | GOR: {gor} scf/STB | WC: {water_cut*100:.0f}%</sup>",
    xaxis_title="<b>Flow Rate (STB/day)</b>",
    yaxis_title="<b>Flowing Bottom-Hole Pressure — FBHP (psia)</b>",
    template="plotly_dark",
    hovermode="x unified",
    legend=dict(x=0.65, y=0.95, bgcolor="rgba(0,0,0,0.5)", bordercolor="white", borderwidth=1),
    margin=dict(l=60, r=40, t=80, b=60),
    height=550
)

# Set axes starting from 0
fig.update_xaxes(range=[0, max(op["aof"]*1.05, 500)])
fig.update_yaxes(range=[0, max(reservoir_pressure*1.1, 5000)])

fig.show()


## 🔧 Module 7: Multi-Tubing Sizing Sensitivity Analysis
Compare how different tubing diameters (2-3/8", 2-7/8", 3-1/2", 4", 4-1/2") impact well production and operating points.


In [ ]:
# =====================================================================
# STEP 3: Multi-Tubing Sizing Analysis
# =====================================================================
print("⚙️ Evaluating standard API tubing sizes...")
multi_res = vlp_calc.multi_tubing_vlp(STANDARD_TUBING_SIZES, max(op["aof"]*1.1, 1000.0), vlp_correlation, n_points=60)

fig_multi = go.Figure()

# Plot IPR
fig_multi.add_trace(go.Scatter(
    x=q_ipr, y=pwf_ipr, mode='lines', name=f'IPR ({ipr_model.upper()})',
    line=dict(color='#00E5FF', width=3.5, dash='dash')
))

# Colors for tubing sizes
colors = ['#FF8A80', '#FFD180', '#FFFF8D', '#B9F6CA', '#84FFFF']

for idx, (label, t_data) in enumerate(multi_res.items()):
    q_arr, fbhp_arr = t_data
    id_in = STANDARD_TUBING_SIZES[label]

    fig_multi.add_trace(go.Scatter(
        x=q_arr, y=fbhp_arr, mode='lines', name=f'VLP: {label}',
        line=dict(color=colors[idx % len(colors)], width=2.5)
    ))

    # Calculate operating point for this tubing size
    vlp_temp = VLPCalculator(fp, tvd, md, id_in, whp, reservoir_temp, surface_temp, roughness)
    s_temp = NodalAnalysis(ipr_calc, vlp_temp)
    op_temp = s_temp.operating_point(ipr_model, vlp_correlation)

    if op_temp["found"]:
        fig_multi.add_trace(go.Scatter(
            x=[op_temp["q"]], y=[op_temp["Pwf"]], mode='markers', name=f'Op Point ({id_in} in)',
            marker=dict(color=colors[idx % len(colors)], size=10, line=dict(color='white', width=1)),
            showlegend=False
        ))

fig_multi.update_layout(
    title="<b>Multi-Tubing Sizing Comparison</b><br><sup>Evaluating Production Rates Across Standard API Tubing Diameters</sup>",
    xaxis_title="<b>Flow Rate (STB/day)</b>",
    yaxis_title="<b>FBHP (psia)</b>",
    template="plotly_dark",
    hovermode="x unified",
    legend=dict(x=0.65, y=0.95, bgcolor="rgba(0,0,0,0.5)", bordercolor="white", borderwidth=1),
    margin=dict(l=60, r=40, t=80, b=60),
    height=550
)
fig_multi.update_xaxes(range=[0, max(op["aof"]*1.05, 500)])
fig_multi.update_yaxes(range=[0, max(reservoir_pressure*1.1, 5000)])
fig_multi.show()


## 🌡️ Module 8: Wellbore Pressure Profile & WHP Sensitivity
Visualize how pressure changes from the surface wellhead down to the reservoir perforations, and analyze how adjusting choke / wellhead pressure impacts production.


In [ ]:
# =====================================================================
# STEP 4: Wellbore Pressure Profile & WHP Sensitivity
# =====================================================================
fig_sub = make_subplots(rows=1, cols=2, subplot_titles=(
    f"<b>Wellbore Pressure Profile</b> (at q = {op['q']:.0f} STB/d)",
    "<b>Wellhead Pressure (WHP) Sensitivity</b>"
))

# ── Left Plot: Pressure Profile vs. Depth ──
if op["q"] > 0:
    profile_res = vlp_calc.compute_fbhp(op["q"], vlp_correlation)
    depths = [p[0] for p in profile_res["profile"]]
    pressures = [p[1] for p in profile_res["profile"]]

    fig_sub.add_trace(go.Scatter(
        x=pressures, y=depths, mode='lines+markers', name='Pressure Profile',
        line=dict(color='#FFAB00', width=3), marker=dict(size=6, color='white')
    ), row=1, col=1)

    fig_sub.update_xaxes(title_text="<b>Pressure (psia)</b>", row=1, col=1)
    fig_sub.update_yaxes(title_text="<b>Measured Depth (ft)</b>", autorange="reversed", row=1, col=1)

# ── Right Plot: WHP Sensitivity ──
whp_test_vals = list(range(50, 851, 100))
whp_q_vals = []
whp_pwf_vals = []

for w_val in whp_test_vals:
    vlp_w = VLPCalculator(fp, tvd, md, tubing_id, w_val, reservoir_temp, surface_temp, roughness)
    s_w = NodalAnalysis(ipr_calc, vlp_w)
    op_w = s_w.operating_point(ipr_model, vlp_correlation)
    whp_q_vals.append(op_w["q"])
    whp_pwf_vals.append(op_w["Pwf"])

fig_sub.add_trace(go.Scatter(
    x=whp_test_vals, y=whp_q_vals, mode='lines+markers', name='Production Rate (q)',
    line=dict(color='#00E676', width=3), marker=dict(size=8, color='white')
), row=1, col=2)

fig_sub.update_xaxes(title_text="<b>Wellhead Pressure — WHP (psia)</b>", row=1, col=2)
fig_sub.update_yaxes(title_text="<b>Flow Rate (STB/day)</b>", row=1, col=2)

fig_sub.update_layout(
    title="<b>Wellbore Hydraulics & Surface Back-Pressure Analysis</b>",
    template="plotly_dark",
    hovermode="x unified",
    showlegend=False,
    margin=dict(l=60, r=40, t=80, b=60),
    height=500
)
fig_sub.show()
